[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/quickstarts/agno_agent_for_evals.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">Agno 旅行智能体追踪项目</h1>

使用 Agno 框架和 OpenAI 模型创建简单的旅行智能体。首先安装必要的 OpenInference 包，并配置 Arize 追踪。

接着定义一组基础工具，提供目的地信息、估算旅行预算并推荐当地活动。

构建并运行基础智能体后，在 Phoenix 中查看追踪输出，理解它如何使用工具和进行推理。

随后按照评估教程完成：
- 配置核心 LLM 并运行内置评估
- 配置使用自定义端点的 LLM
- 创建自定义评估
- 代码评估

需要在终端安装 Arize Phoenix（`pip install arize-phoenix`），并准备 OpenAI API 密钥及免费的 [Tavily](https://auth.tavily.com/) API 密钥。

运行下方单元前，请确保终端中已运行 `phoenix serve`。

## 设置密钥与依赖

In [ ]:
%pip install -qqqqqq arize-phoenix-otel arize-phoenix-evals agno openai openinference-instrumentation-agno openinference-instrumentation-openai httpx

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = globals().get("OPENAI_API_KEY") or getpass(
    "🔑 请输入 OpenAI API 密钥： "
)
os.environ["TAVILY_API_KEY"] = globals().get("TAVILY_API_KEY") or getpass(
    "🔑 请输入 Tavily API 密钥： "
)

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="agno_travel_agent", auto_instrument=True)

## 定义工具

首先定义支持工具的辅助函数，使用 Tavily Search 帮助工具收集各目的地的通用信息。

In [ ]:
# --- 工具辅助函数 ---
import httpx


def _search_api(query: str) -> str | None:
    """优先尝试 Tavily 搜索，失败时返回 None。"""
    tavily_key = os.getenv("TAVILY_API_KEY")
    if not tavily_key:
        return None
    try:
        resp = httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key": tavily_key,
                "query": query,
                "max_results": 3,
                "search_depth": "basic",
                "include_answer": True,
            },
            timeout=8,
        )
        data = resp.json()
        answer = data.get("answer") or ""
        snippets = [r.get("content", "") for r in data.get("results", [])]
        combined = " ".join([answer] + snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None


def _compact(text: str, limit: int = 200) -> str:
    """压缩文本，使输出更简洁。"""
    cleaned = " ".join(text.split())
    return cleaned if len(cleaned) <= limit else cleaned[:limit].rsplit(" ", 1)[0]

智能体可以使用三个工具，后续实验中还会继续增强：

1. Essential Info：提供目的地的天气和一般状况等重要旅行信息。
2. Budget Basics：提供旅行成本信息，根据所选活动帮助规划预算。
3. Local Flavor：推荐独特的当地体验和文化亮点。

In [ ]:
from agno.tools import tool


@tool
def essential_info(destination: str) -> str:
    """获取基础旅行信息，包括天气、最佳时间、景点与礼仪。"""
    q = f"{destination} travel essentials weather best time top attractions etiquette"
    s = _search_api(q)
    if s:
        return f"{destination} essentials: {_compact(s)}"

    return f"{destination}是热门旅行目的地，可以探索当地文化、美食与地标。"


@tool
def budget_basics(destination: str, duration: str) -> str:
    """总结旅行费用类别。"""
    q = f"{destination} travel budget average daily costs {duration}"
    s = _search_api(q)
    if s:
        return f"{destination} budget ({duration}): {_compact(s)}"
    return f"预算： {duration} in {destination}取决于住宿、餐饮、交通与景点。"


@tool
def local_flavor(destination: str, interests: str = "local culture") -> str:
    """推荐地道的当地体验。"""
    q = f"{destination} authentic local experiences {interests}"
    s = _search_api(q)
    if s:
        return f"{destination} {interests}: {_compact(s)}"
    return f"Explore {destination}'s unique {interests}，通过市场、街区与当地餐馆体验。"

## 定义智能体

接下来构建智能体。Agno 框架支持直接定义模型、指令和工具等关键参数，简化构建过程。

In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

trip_agent = Agent(
    name="TripPlanner",
    role="AI Travel Assistant",
    model=OpenAIChat(id="gpt-4.1"),
    instructions=(
        "你是友好且知识丰富的旅行规划师。"
        "综合使用多个工具创建旅行计划，包含必要信息、预算与当地特色。"
        "语气自然清晰，控制在 1000 个词以内。"
    ),
    markdown=True,
    tools=[essential_info, budget_basics, local_flavor],
)

## 运行智能体

现在可以运行智能体！执行此单元查看示例。

In [ ]:
# --- 使用示例 ---
destination = "Tokyo"
duration = "5 days"
interests = "food, culture"

query = f"""
规划持续 {duration}、前往 {destination} 的旅行。
重点关注： {interests}.
包含必要信息、预算明细与当地体验。
"""
trip_agent.print_response(query, stream=True)

In [ ]:
from phoenix.client import Client

client = Client()
spans_df = client.spans.get_spans_dataframe(project_identifier="agno_travel_agent")
agent_spans = spans_df[spans_df["span_kind"] == "AGENT"]
agent_spans

# 使用内置评估模板与 OpenAI 模型运行评估

首先使用常见的 LLM 配置和内置模板运行首次评估。

In [ ]:
from phoenix.evals.llm import LLM

llm = LLM(
    provider="openai",
    model="gpt-4o",
    client="openai",
)

In [ ]:
from phoenix.evals.metrics import CorrectnessEvaluator

correctness_eval = CorrectnessEvaluator(llm=llm)

print(correctness_eval.describe())

In [ ]:
from phoenix.evals import async_evaluate_dataframe, bind_evaluator

from phoenix.trace import suppress_tracing

bound_evaluator = bind_evaluator(
    evaluator=correctness_eval,
    input_mapping={
        "input": "attributes.input.value",
        "output": "attributes.output.value",
    },
)

with suppress_tracing():
    results_df = await async_evaluate_dataframe(agent_spans, [bound_evaluator], concurrency=10)
print(results_df)

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

evaluations = to_annotation_dataframe(dataframe=results_df)

Client().spans.log_span_annotations_dataframe(dataframe=evaluations)

# 使用内置评估模板与自定义模型运行评估

现在创建自定义 LLM 配置，使用同一评估器重新运行内置模板。

In [ ]:
import os

os.environ["FIREWORKS_API_KEY"] = globals().get("FIREWORKS_API_KEY") or getpass(
    "🔑 请输入 Fireworks API 密钥： "
)

In [ ]:
from phoenix.evals.llm import LLM

custom_llm = LLM(
    provider="openai",
    model="accounts/fireworks/models/qwen3-235b-a22b-instruct-2507",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.environ.get("FIREWORKS_API_KEY"),
)

In [ ]:
from phoenix.evals import async_evaluate_dataframe, bind_evaluator
from phoenix.evals.metrics import CorrectnessEvaluator

from phoenix.trace import suppress_tracing

correctness_eval = CorrectnessEvaluator(llm=custom_llm)

bound_evaluator = bind_evaluator(
    evaluator=correctness_eval,
    input_mapping={
        "input": "attributes.input.value",
        "output": "attributes.output.value",
    },
)

with suppress_tracing():
    results_df = await async_evaluate_dataframe(agent_spans, [bound_evaluator], concurrency=10)
print(results_df)

In [ ]:
evaluations = to_annotation_dataframe(dataframe=results_df)
Client().spans.log_span_annotations_dataframe(dataframe=evaluations)

# 创建自定义评估器

创建自定义评估器，并使用刚才创建的自定义 LLM 配置。继续评估正确性，同时增加更贴近应用的上下文。

In [ ]:
CUSTOM_CORRECTNESS_TEMPLATE = """你是专业评估者，需要判断旅行规划智能体的回复是否正确。该智能体是友好的旅行规划师，必须结合多个工具创建包含以下内容的计划：(1) 必要信息，(2) 预算明细，(3) 当地特色与体验。

CORRECT：回复符合以下条件：
- 准确回应用户的目的地、时长与兴趣
- 包含该目的地必要旅行信息，例如天气、最佳游览时间、主要景点与礼仪
- 提供适合目的地及旅行时长的预算或费用明细
- 包含符合用户兴趣的当地体验、文化亮点或地道推荐
- 事实准确、逻辑一致，有助于规划旅行
- 使用准确、适合旅行场景的术语

INCORRECT：回复包含以下任一问题：
- 目的地、费用或当地信息存在事实错误
- 用户要求完整计划时缺少必要信息
- 预算缺失或不适合给定目的地与时长
- 缺少当地体验，或体验泛泛而谈、不符合用户兴趣
- 回应了错误的目的地、时长或兴趣
- 存在矛盾、误导陈述、无用或偏题内容

[BEGIN DATA]
************
[用户输入]：
{{input}}

************
[旅行计划]：
{{output}}
************
[END DATA]

重点关注旅行计划的事实准确性与完整性，包括必要信息、预算和当地特色。输出是 correct 还是 incorrect？"""

In [ ]:
from phoenix.evals import ClassificationEvaluator

custom_correctness_evaluator = ClassificationEvaluator(
    name="custom_correctness",
    llm=llm,
    prompt_template=CUSTOM_CORRECTNESS_TEMPLATE,
    choices={"correct": 1, "incorrect": 0},
)

In [ ]:
bound_evaluator = bind_evaluator(
    evaluator=custom_correctness_evaluator,
    input_mapping={
        "input": "attributes.input.value",
        "output": "attributes.output.value",
    },
)

with suppress_tracing():
    results_df = await async_evaluate_dataframe(agent_spans, [bound_evaluator], concurrency=10)
print(results_df)

In [ ]:
evaluations = to_annotation_dataframe(dataframe=results_df)
Client().spans.log_span_annotations_dataframe(dataframe=evaluations)